# 07-05 · '위치 방식' 검증 (유동/비유동 구간으로 단기·장기 판정)

| 항목 | 내용 |
|---|---|
| 하는 일 | 재무상태표에서 '유동부채' 머리행과 '비유동부채' 머리행 사이 어디에 차입·사채 행이 있는지로 단기/장기를 판정하고, 계정 이름 접두어·표준 계정코드와 어긋나는지 센다 |
| 읽는 파일 | `outputs/04_본수집/본수집_로그.csv`, 저장된 응답 |
| 만드는 파일 | `outputs/05_계정확보/위치분류_검증_요약.txt`, `위치분류_검증_어긋남_목록.csv` |
| 인증키 | 필요 없음 (저장된 파일만 읽습니다) |
| 걸리는 시간 | 약 1분 12초 — 2026-11-05, 이 컴퓨터에서 측정 |

> 이 노트북이 **원본**입니다. 같은 이름의 `.py`는 `scripts/노트북에서_py_만들기.py`가 여기서 자동으로 만든 파일이라 직접 고치지 않습니다. 원래 손으로 쓴 `.py`는 이 폴더의 `원본_스크립트/`에 보관했습니다.

**위치 방식**은 단기차입금(E의 분자)을 이름이 아니라 재무상태표의 *위치*로 정하는 방법입니다. 금액은 읽지 않으며 ICR·E도 계산하지 않습니다. 차입·사채 행의 정의는 탐색용으로 에이전트가 정했습니다. (이 검증의 비교 규칙은 처음에 잘못 만들어 한 번 고쳤습니다 — 어긋남 817→31건, 160→4건. 경위는 `docs/A_사전고정_문서_초안.md`의 변경 기록에 있습니다.)

### 준비 — 불러오기와 경로

필요한 도구를 불러오고, 이 폴더 위치(`HERE`)를 잡습니다. 맨 위 따옴표 글(독스트링)은 이 스크립트의 원래 사용설명서입니다.

In [1]:
"""재무상태표에서 차입·사채 행의 '단기(유동)/장기(비유동)'를 위치로 판단하는 방법을 전체 응답으로 검증한다.

방법(위치 방식): 재무상태표(BS)에서 '유동부채' 머리행과 '비유동부채' 머리행을 찾고, 차입·사채 행이 그 사이 어디에 있는지로 단기/장기를 판단한다.
검증: 위치 판단을 (1) 계정 이름 접두어(유동·단기 / 장기·비유동), (2) 표준 계정코드(account_id)의 의미와 비교해 어긋남을 센다.
금액은 읽지 않는다. ICR·E를 계산하지 않는다.

차입·사채 행의 정의(탐색용, 에이전트가 정함): 정규화한 계정 이름에 '차입' 또는 '사채'가 있고,
 '리스', '할인발행차금', '상환할증금', '대손', '이자', '미지급', '매입', '채권' 가운데 하나라도 있으면 제외.
실행: python -X utf8 05_위치분류_검증.py (저장된 파일만 읽음, 키 필요 없음)
저장: outputs/05_계정확보/위치분류_검증_요약.txt, 위치분류_검증_어긋남_목록.csv
"""
import csv
import json
import re
import sys
from collections import Counter
from pathlib import Path
try:
    HERE = HERE      # .py로 실행할 때
except NameError:
    HERE = Path.cwd()                           # 노트북에서 실행할 때(노트북이 있는 폴더)
sys.path.insert(0, str(HERE.parent / "01_파일럿"))
import config  # noqa: E402
if hasattr(sys.stdout, "reconfigure"):          # 노트북의 출력 창에는 이 함수가 없다
    sys.stdout.reconfigure(encoding="utf-8")

### 상수 — 입력·출력 경로와 규칙 값

In [2]:
LOG = config.BASE / "outputs" / "04_본수집" / "본수집_로그.csv"
OUTDIR = config.BASE / "outputs" / "05_계정확보"
FS_DIR = config.RAW / "fs"
DEBT = re.compile(r"차입|사채")
EXCL = re.compile(r"리스|할인발행차금|상환할증금|대손|이자|미지급|매입|채권")
# 대소문자를 구분한다: 'NoncurrentPortion…' 안의 소문자 'currentPortion'이 단기로 잘못 잡히지 않게 하려는 것(1차 실행의 규칙 결함을 고침)
ID_C = re.compile(r"Shortterm|ShortTerm|CurrentPortion|CurrentBond|CurrentBorrow|CurrentLoan")
ID_N = re.compile(r"LongTerm|Longterm|Noncurrent|NonCurrent")

#### `norm()`

In [3]:
def norm(s):
    return re.sub(r"\s+", "", s or "")

#### `name_sig()`

In [4]:
def name_sig(nm):
    # '비유동차입금의유동성대체부분', '장기차입금(유동성)'처럼 이름이 장기로 시작해도 1년 안에 갚을 몫이면 단기(1차 실행의 규칙 결함을 고침)
    if re.search(r"유동성|\(유동\)|유동성대체", nm) and not re.match(r"^비유동(?!.*유동성)", nm):
        return "C"
    if re.match(r"^(유동|단기)", nm):
        return "C"
    if re.match(r"^(장기|비유동)", nm):
        return "N"
    return "?"

#### `id_sig()`

In [5]:
def id_sig(aid):
    if aid and not aid.startswith("-"):
        if ID_C.search(aid):
            return "C"
        if ID_N.search(aid):
            return "N"
    return "?"

#### `sections()` — BS 행 목록(ord순)에서 각 행의 구간: C(유동부채 구간) / N(비유동부채 구간) / O(그 밖)

In [6]:
def sections(bs):
    """BS 행 목록(ord순)에서 각 행의 구간: C(유동부채 구간) / N(비유동부채 구간) / O(그 밖)"""
    hdr = []
    for i, r in enumerate(bs):
        nm, aid = norm(r["account_nm"]), r.get("account_id", "")
        if nm == "유동부채" or aid == "ifrs-full_CurrentLiabilities":
            hdr.append((i, "C"))
        elif nm == "비유동부채" or aid == "ifrs-full_NoncurrentLiabilities":
            hdr.append((i, "N"))
        elif nm == "부채총계" or aid == "ifrs-full_Liabilities":
            hdr.append((i, "T"))
    has_c = any(t == "C" for _, t in hdr)
    has_n = any(t == "N" for _, t in hdr)
    sec = {}
    cur = "O"
    hi = 0
    for i, r in enumerate(bs):
        while hi < len(hdr) and hdr[hi][0] <= i:
            cur = {"C": "C", "N": "N", "T": "O"}[hdr[hi][1]]
            hi += 1
        sec[i] = cur
    return sec, has_c, has_n

#### `main()`

In [7]:
def main():
    chosen = {}
    for r in csv.DictReader(open(LOG, encoding="utf-8-sig")):
        if r["status"] == "000":
            k = (r["corp_code"], int(r["year"]), r["report"])
            if k not in chosen or r["fs_div"] == "CFS":
                chosen[k] = r["fs_div"]
    n_resp = n_hdr = 0
    rows_total = Counter()
    conf = []
    for i, ((corp, year, rep), fs) in enumerate(sorted(chosen.items()), 1):
        d = json.loads((FS_DIR / f"{corp}_{year}_{config.REPORTS[rep]}_{fs}.json").read_text(encoding="utf-8"))["list"]
        bs = sorted([x for x in d if x.get("sj_div") == "BS"], key=lambda x: int(x["ord"]))
        n_resp += 1
        sec, has_c, has_n = sections(bs)
        if has_c and has_n:
            n_hdr += 1
        for j, x in enumerate(bs):
            nm = norm(x["account_nm"])
            if not DEBT.search(nm) or EXCL.search(nm):
                continue
            pos = sec[j] if (has_c and has_n) else "?"
            ns, ids = name_sig(nm), id_sig(x.get("account_id", ""))
            rows_total["차입·사채 행"] += 1
            if pos == "?":
                rows_total["머리행 없음으로 위치 판단 불가"] += 1
                continue
            if pos == "O":
                rows_total["부채 구간 밖(유동/비유동 어느 쪽도 아님)"] += 1
                continue
            rows_total["위치로 단기/장기 판단됨"] += 1
            if ns == "?" and ids == "?":
                rows_total["이름·코드로는 판단 불가, 위치로만 판단"] += 1
            for lab, sg in (("이름", ns), ("표준코드", ids)):
                if sg != "?":
                    rows_total[f"{lab} 신호 있음"] += 1
                    if sg != pos:
                        rows_total[f"{lab}과 위치가 어긋남"] += 1
                        if len(conf) < 3000:
                            conf.append({"corp_code": corp, "year": year, "report": rep, "fs": fs, "계정명": nm,
                                         "account_id": x.get("account_id", ""), "위치": pos, "비교": lab, "비교신호": sg})
        if i % 8000 == 0:
            print(f"  {i:,}개 읽음")
    lines = [f"응답 {n_resp:,}개 중 유동부채·비유동부채 머리행이 모두 있는 응답: {n_hdr:,}개 ({n_hdr/n_resp:.1%})"]
    for k in ["차입·사채 행", "머리행 없음으로 위치 판단 불가", "부채 구간 밖(유동/비유동 어느 쪽도 아님)", "위치로 단기/장기 판단됨",
              "이름·코드로는 판단 불가, 위치로만 판단", "이름 신호 있음", "이름과 위치가 어긋남", "표준코드 신호 있음", "표준코드과 위치가 어긋남"]:
        lines.append(f"{k}: {rows_total[k]:,}")
    for lab in ("이름", "표준코드"):
        a, b = rows_total[f"{lab} 신호 있음"], rows_total[f"{lab}과 위치가 어긋남"] + rows_total[f"{lab}과 위치가 어긋남"] * 0
        if a:
            lines.append(f"  {lab} 신호와 위치가 일치한 비율: {1 - rows_total[f'{lab}과 위치가 어긋남']/a:.2%}")
    text = "\n".join(lines)
    print(text)
    OUTDIR.mkdir(parents=True, exist_ok=True)
    (OUTDIR / "위치분류_검증_요약.txt").write_text(text, encoding="utf-8")
    with open(OUTDIR / "위치분류_검증_어긋남_목록.csv", "w", encoding="utf-8-sig", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["corp_code", "year", "report", "fs", "계정명", "account_id", "위치", "비교", "비교신호"])
        w.writeheader()
        w.writerows(conf)
    print("저장:", OUTDIR)

### (.py에만 들어가는 부분)

노트북에서는 `__name__`이 항상 `"__main__"`이라 아래 코드를 일반 셀로 두면 열 때마다 실행됩니다. 그래서 실행하지 않는 설명 셀에 두고, 변환 스크립트가 `.py`에만 넣습니다.

```python
if __name__ == "__main__":
    main()
```

In [8]:
main()

  8,000개 읽음


  16,000개 읽음


  24,000개 읽음


  32,000개 읽음


응답 34,964개 중 유동부채·비유동부채 머리행이 모두 있는 응답: 34,808개 (99.6%)
차입·사채 행: 82,697
머리행 없음으로 위치 판단 불가: 247
부채 구간 밖(유동/비유동 어느 쪽도 아님): 62
위치로 단기/장기 판단됨: 82,388
이름·코드로는 판단 불가, 위치로만 판단: 10,054
이름 신호 있음: 65,652
이름과 위치가 어긋남: 31
표준코드 신호 있음: 68,867
표준코드과 위치가 어긋남: 4
  이름 신호와 위치가 일치한 비율: 99.95%
  표준코드 신호와 위치가 일치한 비율: 99.99%
저장: C:\Users\SAMSUNG\OneDrive\바탕 화면\공모전_26-2 BAF\BAF-26-2-finance_2\BAF-26-2-finance_2\Corp\outputs\05_계정확보


### 저장된 결과 미리보기 (맨 위 5행)

위 실행이 만들었거나 이미 저장돼 있던 결과 파일을 표로 엽니다. 파일 전체는 엑셀로 여세요.

In [9]:
import pandas as pd
from IPython.display import display

_p = HERE.parents[1] / "outputs/05_계정확보/위치분류_검증_어긋남_목록.csv"
print(_p.name, "|", end=" ")
_df = pd.read_csv(_p, encoding="utf-8-sig", low_memory=False)
print(f"{len(_df):,}행 × {_df.shape[1]}열")
display(_df.head(5))

위치분류_검증_어긋남_목록.csv | 35행 × 9열


,corp_code,year,report,fs,계정명,account_id,위치,비교,비교신호
0,120571,2023,FY,CFS,비유동차입금의비유동성부분,-표준계정코드 미사용-,N,이름,C
1,120571,2024,H1,CFS,비유동차입금의비유동성부분,-표준계정코드 미사용-,N,이름,C
2,120571,2024,Q1,CFS,비유동차입금의비유동성부분,-표준계정코드 미사용-,N,이름,C
3,120571,2024,Q3,CFS,비유동차입금(사채포함)의비유동성부분,ifrs-full_LongtermBorrowings,N,이름,C
4,121941,2023,FY,CFS,비유동차입금의비유동성부분,ifrs-full_LongtermBorrowings,N,이름,C
